# Protein-protein binder design
*How to design proteins that bind another protein*

In this tutorial, we'll demonstrate how to use the OpenProtein.AI Python client to
design a protein that binds another protein. We refer to the designed protein as the
**binder** and the protein being bound as the **target**.

TODO: improve diagram TODO: adjust sizing, whitespace

<img src="../_static/walkthroughs/protein_protein_binder_design/design_problem.png" width="500"/>

The design process consists of four main steps:

1. **Query Specification**: Specify the design problem as a "query", including

    1. the target protein
    1. the specific epitope that we want to target
    1. the length of binder

1. **Structure Generation**: Generate plausible structures for the binder, using
   RFdiffusion.

1. **Sequence Design**: Design corresponding sequences for each binder structure,
   using ProteinMPNN.

1. **In Silico Validation**: Validate the designed sequences by predicting their
   structures and computing in silico metrics that are indicative of expression and
   binding. Filter and select designs for experimental evaluation based on the metrics.

TODO: consider mentioning benefits of our unified query specification api in the intro

## Prerequisites

TODO: link to page regarding credentials set up

To run this tutorial, you'll need a Python environment containing the following
packages:

- `openprotein_python>=0.9.1`
- `molviewspec` (for structure visualization)

Additionally, you should have your credentials set up in `~/.openprotein/config.toml` to
authenticate with the OpenProtein.AI API. Below, we

- import the necessary packages, and
- connect to the OpenProtein.AI API

### Import necessary packages

In [162]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import numpy.typing as npt
import pandas as pd
from scipy.spatial.transform import Rotation

from tqdm import tqdm

import molviewspec as mvs
from molviewspec.nodes import RepresentationTypeT

import openprotein
from openprotein import Protein, MolecularComplex

### Connect to OpenProtein.AI

In [163]:
session = openprotein.connect()
print("✅ Successfully connected to the OpenProtein.AI API!")

✅ Successfully connected to the OpenProtein.AI API!


## Step 1: Query Specification
*Specify the protein-protein binder design problem*

In this tutorial, we focus on designing a binder for **Interleukin-7 receptor alpha 
(IL-7Rα)**, a key target in the human immune system. This design problem is 
adapted from the **RFdiffusion** study ([Watson et al., 2023](https://www.nature.com/articles/s41586-023-06415-8)),
which demonstrated the de novo design of high-affinity binders to this target.

In this step, we will create an object, which we refer to as the **query**, that
represents this design problem. The query will be a `MolecularComplex` object containing
one `Protein` chain representing the target, and one `Protein` chain representing the
binder. We will use these classes (`MolecularComplex` and `Protein`) to create a query that specifies:

1. **Target:** The sequence and structure of the IL-7Rα extracellular domain.
1. **Epitope:** The specific residues of IL-7Rα where the binder should bind.
1. **Binder Length:** The desired length of the de novo protein.

## Step 1.1: Specify the target

For the IL-7Rα target, we use the structure from RCSB PDB entry 3DI3. We use the helper
method `Protein.from_pdb_id` to download and parse the structure into a `Protein`
object; the `Protein` class provides a convenient interface for handling the sequence
and structure of a single protein chain.

In [164]:
# Load the IL-7Rα chain, which is chain B
target = Protein.from_pdb_id(pdb_id="3DI3", chain_id="B")
print("target name:", target.name)
print("target sequence:", target.sequence.decode())
print("target length:", len(target))

target name: 3DI3
target sequence: GSHMESGYAQNGDLEDAELDDYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTPEINNSSGEMD
target length: 223


Before we continue, we note that although our `target` protein has 223 residues, the
structure at some of these residues is *undefined*. That is, for some of the residues,
the coordinates of their atoms is unknown.

TODO: update explanation here

To see which residues have undefined structure, we can use the method
`Protein.get_structure_mask`. This returns a boolean array that is `True` where the
structure is undefined and `False` where it is defined.

To help visualize exactly where these undefined regions occur relative to the sequence,
we define a helper function below to help with making the visualization. It's not
necessary to understand the implementation details.

Now, let's use our helper function, `print_aligned_tracks`, to visualize our `target`.
In the visualization below, residues with **undefined structure** are marked with a
**caret** (`^`).

In [165]:
# Visualize the target sequence and its structure mask
target.print(include=("sequence", "structure_mask"))

0     SEQUENCE       GSHMESGYAQNGDLEDAELDDYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIET
0     STRUCTURE_MASK ^^^^^^^^^^^^^^^^^^^^                                                            

80    SEQUENCE       KKFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDEN
80    STRUCTURE_MASK                                                                                 

160   SEQUENCE       KWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTPEINNSSGEMD
160   STRUCTURE_MASK                                                      ^^^^^^^^^^


From the visualization, we can see that the some of the residues at the beginning and
end of the target sequence have undefined structure.

To ensure that the design process focuses only on well-defined regions, we should remove
these residues with undefined structure from our `target` object. But before we do that,
we'll define the epitope first below, as it's slightly easier to identify the binding
site before the sequence is modified.

## Step 1.2: Specify the epitope

The epitope is the set of residues in the target that we want our designed binder to
bind to. We'll use the residues at positions 62, 84, and 143 as our epitope; these are
the same as the residues used in the RFdiffusion study.

TODO: write the appendix, or links to another doc page

NOTE: When using the API, residues are numbered starting from `1`, which follows the
canonical mmcif residue id system (`label_seq_id`), and not the author id system
(`auth_seq_id`). See the Appendix for more information.

To specify the epitope, we use the method `Protein.set_binding_at`. This method is used
to specify which residues are the *binding* residues i.e. the residues that should bind
to another protein chain. In this case, that other chain is the binder we're designing.

In [166]:
binding_sites = [62, 84, 143]
target = target.set_binding_at(binding_sites, value="B")

After defining the binding sites, it is helpful to verify their location relative to the
protein sequence. We can use our `print_aligned_tracks` helper again to visualize the
epitope. In the visualization below, residues marked with `B` are part of the defined
epitope.

In [167]:
target.print(include=("sequence", "binding", "structure_mask"))

0     SEQUENCE       GSHMESGYAQNGDLEDAELDDYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIET
0     BINDING                                                                     B                  
0     STRUCTURE_MASK ^^^^^^^^^^^^^^^^^^^^                                                            

80    SEQUENCE       KKFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDEN
80    BINDING           B                                                          B                 
80    STRUCTURE_MASK                                                                                 

160   SEQUENCE       KWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTPEINNSSGEMD
160   BINDING                                                                       
160   STRUCTURE_MASK                                                      ^^^^^^^^^^


### Remove residues with undefined structure

Now let's remove the residues with undefined structure that we discussed earlier. We
can do this simply by slicing the `target` protein using numpy-like slicing syntax,
keeping only positions where the structure mask is `False`.

In [168]:
print("target length (old):", len(target))
target = target[~target.get_structure_mask()]
print("target length (new):", len(target))

target length (old): 223
target length (new): 193


Let's verify by visualizing our target again:

In [169]:
target.print(include=("sequence", "binding", "structure_mask"))

0     SEQUENCE       DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSL
0     BINDING                                                 B                     B                
0     STRUCTURE_MASK                                                                                 

80    SEQUENCE       TCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQ
80    BINDING                                                  B                                     
80    STRUCTURE_MASK                                                                                 

160   SEQUENCE       PAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTP
160   BINDING                                         
160   STRUCTURE_MASK                                  


Our visualization shows that regions with undefined structure have now been removed!

Note that by truncating the `target`, the positions of our binding site have changed.
This is why we chose to annotate the binding site first - so that we could simply use
the binding site positions aligned to the original target sequence. We can check the
positions of the new binding site by looking at the new binding array:

In [170]:
# NB: we add one here to get 1-indexed positions
binding_sites = np.where(target.get_binding() == "B")[0] + 1
print("binding site:", binding_sites)

binding site: [ 42  64 123]


### Visualize

It's always a good idea to visualize the 3D structures of our `Protein`s to ensure that
we've created them correctly. First, let's define a helper function using the
`molviewspec` package to help us create the visualization. It's not necessary to
understand the implementation details.

In [171]:
@dataclass(frozen=True)
class ColorSpec:
    chain_id: str
    color: str
    positions: list[int] | None = None
    rep_type: RepresentationTypeT = "cartoon"


def visualize_cif(cif_string: str, colors: list[ColorSpec]):
    builder = mvs.create_builder()
    model = (
        builder.download(url="structure.cif").parse(format="mmcif").model_structure()
    )
    for color_spec in colors:
        component = model.component(
            selector=(
                mvs.ComponentExpression(label_asym_id=color_spec.chain_id)
                if color_spec.positions is None
                else [
                    mvs.ComponentExpression(
                        label_asym_id=color_spec.chain_id, label_seq_id=i
                    )
                    for i in color_spec.positions
                ]
            )
        )
        rep = component.representation(type=color_spec.rep_type)
        rep.color(color=color_spec.color)
    builder.molstar_notebook(
        data={"structure.cif": cif_string},
        width=600,
        height=500,
    )

Before we visualize our `target`, we rotate it using the `.transform` method so that
the binding site is easy to see in the visualization:

In [172]:
# 1. Define the rotation matrix `R`
#    (it's not necessary to understand how we define the rotation matrix `R`)
axis, angle = np.array([-1.0, 1.0, 0.0]), np.radians(45)
R = Rotation.from_rotvec(axis / np.linalg.norm(axis) * angle).as_matrix()
# 2. Apply the rotation with `.transform`
target = target.transform(R=R)

Now, let's use the helper function, `visualize_cif`, to visualize our `target`.

In [173]:
visualize_cif(
    cif_string=target.to_string(),
    colors=[
        # color the target chain a light blue
        ColorSpec(chain_id="A", color="#b5e2f5"),
        # color the epitope green, and use the ball_and_stick representation
        ColorSpec(
            chain_id="A",
            color="#6bb50a",
            positions=binding_sites,
            rep_type="ball_and_stick",
        ),
    ],
)

<IPython.core.display.Javascript object>

The visualization shows the structure of IL-7Rα, and the epitope comprising three
residues spread across three adjacent loop regions, as expected!

## Step 1.3: Specify the binder length

For this tutorial, we'll generate binders of length `80`. Since the binder is what we're
designing, its sequence and structure are unknown. To represent this as a `Protein`
object, we simply need to create a `Protein` of length 80, whose sequence
and structure are both undefined! The method `Protein.from_expr` allows us to do this
easily; this method constructs `Protein`s with undefined sequence and structure from 
an **expr**ession describing its length. For our binder of length `80`, the expression
is simply `80`.

In [174]:
binder = Protein.from_expr(80)
print("binder length:", len(binder))

binder length: 80


Let's visualize the `binder`'s sequence and structure mask to check that they are indeed
undefined. Note that unknown residues in a sequence are represented as `X`.

In [175]:
binder.print(include=("sequence", "structure_mask"))

0     SEQUENCE       XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
0     STRUCTURE_MASK ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


As expected, the sequence and structure are both fully undefined.

Finally, now that we have our `target` and `binder`, we can combine them into a
`MolecularComplex` to form our query! We can do so using the `MolecularComplex`
constructor, or more simply, by joining the two `Protein`s using an `&` like we do below.
When combining with `&`, chain ids are assigned alphabetically, from left to right.

In [176]:
query = target & binder
print("Query type", type(query))
print("Chains in query:", list(query.get_proteins().keys()))
print("\nVisualize target (Chain A):")
query.get_protein(chain_id="A").print(include=("sequence", "structure_mask"))
print("\nVisualize binder (Chain B):")
query.get_protein(chain_id="B").print(include=("sequence", "structure_mask"))

Query type <class 'openprotein.molecular_complex.MolecularComplex'>
Chains in query: ['A', 'B']

Visualize target (Chain A):
0     SEQUENCE       DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSL
0     STRUCTURE_MASK                                                                                 

80    SEQUENCE       TCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQ
80    STRUCTURE_MASK                                                                                 

160   SEQUENCE       PAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTP
160   STRUCTURE_MASK                                  

Visualize binder (Chain B):
0     SEQUENCE       XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
0     STRUCTURE_MASK ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


We can now use our `query` to generate binder designs.

# Step 2: Structure Generation
*Generate plausible structures for the binder*

In this step, we sample plausible 3D backbones for the binder that geometrically
complement the target.

## Generate structures with RFdiffusion

In this tutorial, we use RFdiffusion, a popular structure generation method, to generate
the structures. To generate the structures using RFdiffusion, we use the function
`session.models.rfdiffusion.generate`, passing in (1) our `query`, (2) desired number of
structures (`N_STRUCTURES=100`), and (3) any additional arguments to control the
RFdiffusion algorithm.

Below, we run the RFdiffusion job and wait for it to finish; this generally takes about
an hour.

In [126]:
# N_STRUCTURES = 100

# # 1. Create the job
# rfdiffusion_job = session.models.rfdiffusion.generate(
#     query=query,
#     N=N_STRUCTURES,
#     # Following Bennett et al. (2023), we reduce the noise added during
#     # generation, which has been found to help with binder design, albeit at
#     # the cost of some diversity.
#     **{"denoiser.noise_scale_ca": 0.5, "denoiser.noise_scale_frame": 0.5},
# )
# print(rfdiffusion_job)

# # 2. Wait for the job to finish
# MINUTES = 60  # seconds per minute
# _ = rfdiffusion_job.wait_until_done(verbose=True, timeout=60 * MINUTES)

In [127]:
# TODO: remove
N_STRUCTURES = 100
rfdiffusion_job = session.load_job("c9949f53-93c5-4da4-8b49-41b3b6052153")

Next, we retrieve the generated structures as a list of `MolecularComplex` objects.

In [128]:
generated_structures: list[MolecularComplex] = rfdiffusion_job.get()
assert len(generated_structures) == N_STRUCTURES
print("# of structures generated", len(generated_structures))

# of structures generated 100


Each generated structure is similar to our query, except that the backbone structure of
the binder has been filled in by RFdiffusion. Note that the sequence of the binder
remains unknown, as we have only generated its backbone structure. Let's check that this
is the case for the first generated structure:

In [129]:
first_structure = generated_structures[0]
print("Chains in structure:", sorted(first_structure.get_proteins().keys()))
first_target = first_structure.get_protein(chain_id="A")
first_binder = first_structure.get_protein(chain_id="B")
print("\nVisualize first target (Chain A):")
first_target.print(include=("sequence", "structure_mask"))
print("\nVisualize first binder (Chain B):")
first_binder.print(include=("sequence", "structure_mask"))

Chains in structure: ['A', 'B']

Visualize first target (Chain A):
0     SEQUENCE       DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSL
0     STRUCTURE_MASK                                                                                 

80    SEQUENCE       TCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQ
80    STRUCTURE_MASK                                                                                 

160   SEQUENCE       PAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTP
160   STRUCTURE_MASK                                  

Visualize first binder (Chain B):
0     SEQUENCE       XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
0     STRUCTURE_MASK                                                                                 


As expected, the only thing that has changed from the query is that the binder structure
is now fully defined rather than fully undefined, indicating that the binder structure
has been generated.

## Visualize the generated structures

Before proceeding, it's important to visually inspect some of the generated structures
to check that they look reasonable and satisfy our query specification.

Visualizing the first structure below, we see that the target-binder complex does indeed
satisfy our query specification: the binder is positioned close to the target epitope we
specified, and has the right length.

In [130]:
visualize_cif(
    # TODO: need transform with old rfdiffusion job, but won't need with new job
    generated_structures[0].copy().transform(R=R).to_string(),
    colors=[
        ColorSpec(chain_id="A", color="#b5e2f5"),  # target in blue
        ColorSpec(chain_id="B", color="#f4c30b"),  # binder in orange
        ColorSpec(
            chain_id="A",
            color="#6bb50a",  # epitope in green
            positions=binding_sites,
            rep_type="ball_and_stick",
        ),
    ],
)

<IPython.core.display.Javascript object>

Looking at another structure below, we see that it also satisfies our query
specification. It has a different fold from the first structure, indicating that
there is also *diversity* in the generated binder backbones. *Diversity* in generated
structures is critical for maximizing success, as it increases the effective number of
independent hypotheses explored.

In [131]:
visualize_cif(
    # TODO: need transform with old rfdiffusion job, but won't need with new job
    generated_structures[88].copy().transform(R=R).to_string(),
    colors=[
        ColorSpec(chain_id="A", color="#b5e2f5"),  # target in blue
        ColorSpec(chain_id="B", color="#f4c30b"),  # binder in orange
        ColorSpec(
            chain_id="A",
            color="#6bb50a",  # epitope in green
            positions=binding_sites,
            rep_type="ball_and_stick",
        ),
    ],
)

<IPython.core.display.Javascript object>

# Step 3: Sequence Design
*Design sequences for the binder*

In this step, we design sequences for each binder backbone structure using an inverse
folding model. We'll design multiple sequences per structure to increase the chance of
designing a sequence that folds into the desired structure.

## Design sequences with ProteinMPNN

In this tutorial, we use ProteinMPNN to design the binder sequences. To generate
sequences using ProteinMPNN, we use the function `session.models.proteinmpnn.generate`,
passing in (1) the structure generated by RFdiffusion, (2) the desired number of
sequences per structure (`N_SEQS_PER_STRUCTURE=10`), and (3) any additional arguments to
control the ProteinMPNN algorithm.

Below, we run a ProteinMPNN job for every generated structure and wait for the jobs to
finish; this generally takes about 15 minutes.

In [132]:
# N_SEQS_PER_STRUCTURE = 10

# # 1. Create the jobs
# proteinmpnn_jobs = []
# for generated_structure in tqdm(
#     generated_structures, mininterval=1.0, desc="Creating jobs"
# ):
#     proteinmpnn_job = session.models.proteinmpnn.generate(
#         query=generated_structure,
#         num_samples=N_SEQS_PER_STRUCTURE,
#         temperature=0.1,
#         seed=42,  # for reproducibility
#     )
#     proteinmpnn_jobs.append(proteinmpnn_job)

# # 2. Wait for the jobs to finish
# for proteinmpnn_job in tqdm(proteinmpnn_jobs, mininterval=1.0, desc="Waiting for jobs"):
#     _ = proteinmpnn_job.wait_until_done(timeout=15 * MINUTES)
#     assert proteinmpnn_job.status == "SUCCESS"

In [133]:
# TODO: remove
import json
N_SEQS_PER_STRUCTURE = 10
proteinmpnn_job_ids_cache_path = Path("data/outputs/3DI3_binder_designs/proteinmpnn_job_ids.json")
# proteinmpnn_job_ids_cache_path.write_text(json.dumps([j.id for j in proteinmpnn_jobs]))
proteinmpnn_jobs = [session.load_job(x) for x in json.load(proteinmpnn_job_ids_cache_path.open("rb"))]

In [134]:
# TODO: remove
# 2. Wait for the jobs to finish
MINUTES = 60
for proteinmpnn_job in tqdm(proteinmpnn_jobs, mininterval=1.0, desc="Waiting for jobs"):
    _ = proteinmpnn_job.wait_until_done(timeout=15 * MINUTES)
    assert proteinmpnn_job.status == "SUCCESS"

Waiting for jobs: 100%|██████████| 100/100 [00:05<00:00, 18.65it/s]


TODO: break up this section?

Each ProteinMPNN job returns the designed sequences for the corresponding structure. The
designed sequence includes the sequence of both the target and the binder in a colon
separated string, ordered by chain id i.e.. `<target_sequence>:<binder_sequence>`.

The sequences of the target returned by ProteinMPNN should always be same as the
sequence of the original target as we do not want to redesign the target. On the other
hand, the designed sequences of the binders should be novel sequences rather than fully
masked sequences of all `X`.

Let's retrieve the results of the first ProteinMPNN job to verify that this is the case.

In [135]:
first_proteinmpnn_results = proteinmpnn_jobs[0].get()
# inspect first sequence
first_design_seq = first_proteinmpnn_results[0].sequence
first_target_seq, first_binder_seq = first_design_seq.split(":")
assert first_target_seq == target.sequence.decode(), "should match original target"
print("first target sequence", first_target_seq)
print("first binder sequence", first_binder_seq)
# inspect second sequence
second_design_seq = first_proteinmpnn_results[1].sequence
second_target_seq, second_binder_seq = second_design_seq.split(":")
assert second_target_seq == target.sequence.decode(), "should match original target"
print("second target sequence", second_target_seq)
print("second binder sequence", second_binder_seq)

first target sequence DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTP
first binder sequence SLKKKIEELKKKAEKAEVELKKVEADKKVLEKVLEAKAKLYPEKKEEIEKEKAKLTAEYEKKLEELKKEIEEYKKKIKEL
second target sequence DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKCLNFRKLQEIYFIETKKFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKDENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTP
second binder sequence SLEEKKKELEEKAKKAKEELKKVEAEMKVLKAVLEAKAKLYPEKAEEYKKELEEKTAEYEAKIKELEEKIKEYEEKLKKL


Each designed sequence is also associated with a score, which indicates the likelihood
of the sequence. This score is sometimes used in the in silico validation step, although
we do not use it in this tutorial.

In [136]:
first_design_score = first_proteinmpnn_results[0].score.item()
second_design_score = first_proteinmpnn_results[1].score.item()
print("first design score", first_design_score)
print("second design score", second_design_score)

first design score 1.0672
second design score 1.0364


Let's collect all of the designs into a dataframe for further analysis.

In [137]:
records = []
for i, proteinmpnn_job in enumerate(tqdm(proteinmpnn_jobs, mininterval=1.0)):
    for j, proteinmpnn_result in enumerate(proteinmpnn_job.get()):
        records.append(
            {
                "design_idx": i * N_SEQS_PER_STRUCTURE + j,
                "structure_idx": i,
                "sequence_idx": j,
                "score": proteinmpnn_result.score.item(),
                "sequence": proteinmpnn_result.sequence,
            }
        )
df = pd.DataFrame.from_records(records).set_index(["structure_idx", "sequence_idx"])
df.head()

100%|██████████| 100/100 [00:13<00:00,  7.58it/s]


design_idx   score  \
structure_idx sequence_idx                       
0             0                      0  1.0672   
              1                      1  1.0364   
              2                      2  1.0173   
              3                      3  1.0845   
              4                      4  1.0763   

                                                                     sequence  
structure_idx sequence_idx                                                     
0             0             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...  
              1             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...  
              2             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...  
              3             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...  
              4             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...

# Step 4: In Silico Validation
*Validate, filter, and select designs using structure prediction*

In the last step, we validate our designs by predicting the structure of our designed
sequences to

1. filter for sequences that are likely to adopt the desired structure, and
2. select promising designs for experimental evaluation based on metrics computed from
   the predicted structures; these metrics are highly correlated with expression and
   binding across a variety of previously engineered binders.

## Predict structures with Boltz-2

We predict the structures of our designs using Boltz-2.

Since the structure of the target is known, it is typical to predict the target in
single sequence mode (i.e. no MSA), and to use the known structure of the target as its
template. Since we do not support templates yet (coming soon!), we'll use an MSA for
the target instead; this generally achieves the same objective, allowing the model to
accurately recapitulate the target's known structure.

On the other hand, since the binder sequence is novel, it is typically predicted in
single sequence mode without a template to avoid introducing evolutionary biases.

Below, we compute an MSA for the target, and use it to predict the structure of all
designed sequences; this generally takes about 45 minutes.

In [138]:
# # 1. Compute MSA for target to use for folding
# target_msa = session.align.create_msa(target.sequence)

# # 2. Create the complexes to fold
# complexes_to_fold = []
# for i, generated_structure in enumerate(generated_structures):  # for each structure
#     proteinmpnn_job = proteinmpnn_jobs[i]
#     for proteinmpnn_result in proteinmpnn_job.get():  # for each sequence
#         _, designed_binder_sequence = proteinmpnn_result.sequence.split(":")
#         complex = MolecularComplex(
#             proteins={
#                 "A": Protein(target.sequence),
#                 "B": Protein(designed_binder_sequence),
#             },
#         )
#         # set MSAs to use for structure prediction
#         complex.proteins["A"].msa = target_msa
#         complex.proteins["B"].msa = Protein.single_sequence_mode
#         complexes_to_fold.append(complex)

# # 3. Create the job
# fold_job = session.fold.boltz_2.fold(complexes_to_fold)
# print(fold_job)

# # 4. Wait for the job to finish
# _ = fold_job.wait_until_done(verbose=True, timeout=60 * MINUTES)

In [139]:
# TODO: remove
fold_job = session.load_job("1924f2f6-1321-4a9d-908d-c1ed9358b28e")
fold_job

FoldJob(num_records=None, job_id='1924f2f6-1321-4a9d-908d-c1ed9358b28e', job_type=<JobType.embeddings_fold: '/embeddings/fold'>, status=<JobStatus.SUCCESS: 'SUCCESS'>, created_date=datetime.datetime(2026, 1, 11, 3, 28, 28, 337325, tzinfo=TzInfo(0)), start_date=datetime.datetime(2026, 1, 11, 3, 28, 34, 906358, tzinfo=TzInfo(0)), end_date=datetime.datetime(2026, 1, 11, 4, 5, 43, 77534, tzinfo=TzInfo(0)), prerequisite_job_id='4c0bfd6c-a25d-460a-ad7a-01dbc67d9ef7', progress_message=None, progress_counter=100, sequence_length=None)

We retrieve the predicted structures as follows:

In [140]:
predicted_structures: list[MolecularComplex] = fold_job.get(verbose=True)

Retrieving: 100%|██████████| 1000/1000 [00:22<00:00, 45.07it/s]


Let's visualize the first predicted structure to check that it looks reasonable:

In [141]:
visualize_cif(
    predicted_structures[0]
    .copy()
    .transform(
        # apply a rotation to make the binding site clearer
        # (its not necessary to understand how to define the rotation)
        R=(
            Rotation.from_euler("y", 180, degrees=True)
            * Rotation.from_euler("x", -90, degrees=True)
        ).as_matrix()
    )
    .to_string(),
    colors=[
        ColorSpec(chain_id="A", color="#b5e2f5"),  # target in blue
        ColorSpec(chain_id="B", color="#f4c30b"),  # binder in orange
        ColorSpec(
            chain_id="A",
            color="#6bb50a",  # epitope in green
            positions=binding_sites,
            rep_type="ball_and_stick",
        ),
    ],
)

<IPython.core.display.Javascript object>

As expected, the predicted structure contains the target, and the binder close to the
target chain.

In addition to the predicted structures, we also retrieve the predicted aligned errors
(PAEs), which we will use for computing metrics below. The [PAE](https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/pae-a-measure-of-global-confidence-in-alphafold-predictions/) is a structure prediction
confidence metric that has been [highly effective at identifying successful binders](https://www.nature.com/articles/s41467-023-38328-5).

In [142]:
predicted_paes: list[npt.NDArray[np.floating]] = fold_job.get_pae()

## Filter and select designs by metrics

Following standard practice, we compute the following metrics:

| Metric | Description | Ideal Value |
| --- | --- | --- |
| **RMSD** | Measures how closely the predicted structure of the *entire complex* matches the generated structure. | < 2.5 Å |
| **iPAE** | Confidence that the binder forms an interface with the target. | < 10 |
| **Binder RMSD** | Measures how closely the predicted structure of *just the binder* matches the generated structure. | < 1 Å |
| **Binder pLDDT** | Confidence in the predicted structure of the binder. | > 80 |

### Compute Metrics

Below, we compute these metrics and collate the metrics and designed sequences into a
dataframe for further analysis.

In [143]:
records = []  # collect metrics and designed sequences into a list of records
for i, generated_structure in enumerate(tqdm(generated_structures, mininterval=1.0)):
    for j in range(N_SEQS_PER_STRUCTURE):
        predicted_structure = predicted_structures[i * N_SEQS_PER_STRUCTURE + j]
        # compute overall rmsd
        rmsd = predicted_structure.rmsd(generated_structure)
        # compute ipae
        pae = predicted_paes[i * N_SEQS_PER_STRUCTURE + j].squeeze(0)
        ipae0 = np.mean(pae[: len(target), len(target) :])
        ipae1 = np.mean(pae[len(target) :, : len(target)])
        ipae = (ipae0 + ipae1) / 2
        # compute binder metrics
        generated_binder = generated_structure.get_protein(chain_id="B")
        predicted_binder = predicted_structure.get_protein(chain_id="B")
        binder_rmsd = predicted_binder.rmsd(generated_binder)
        binder_plddt = predicted_binder.plddt.mean()
        # get dataframe row containing designed sequence
        row = df.loc[(i, j)]
        # record all relevant data
        records.append(
            {
                "design_idx": row["design_idx"],
                "structure_idx": i,
                "sequence_idx": j,
                "rmsd": rmsd,
                "ipae": ipae,
                "binder_rmsd": binder_rmsd,
                "binder_plddt": binder_plddt,
                "score": row["score"],
                "sequence": row["sequence"],
            }
        )
df = pd.DataFrame.from_records(records).set_index(["structure_idx", "sequence_idx"])
df.head()

100%|██████████| 100/100 [00:00<00:00, 111.02it/s]


design_idx       rmsd       ipae  binder_rmsd  \
structure_idx sequence_idx                                                  
0             0                      0  19.908426  11.530317     0.791237   
              1                      1  21.301144  10.755597     0.559875   
              2                      2  19.710243   5.653427     0.584873   
              3                      3  18.583298  20.554720     1.034244   
              4                      4  14.432941  21.446678     0.468565   

                            binder_plddt   score  \
structure_idx sequence_idx                         
0             0                88.083885  1.0672   
              1                85.622124  1.0364   
              2                92.960564  1.0173   
              3                91.049095  1.0845   
              4                91.137978  1.0763   

                                                                     sequence  
structure_idx sequence_idx                                                     
0             0             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...  
              1             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...  
              2             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...  
              3             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...  
              4             DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...

### Filter and select designs by metrics

We start by filtering the designs based on the ideal metric thresholds.

In [144]:
df_filtered = df[
    (df["rmsd"] < 2.5)
    & (df["ipae"] < 10)
    & (df["binder_rmsd"] < 1)
    & (df["binder_plddt"] > 80)
]
print("# designs passing filters", len(df_filtered))
print(
    "# unique structures passing filters",
    df_filtered.index.get_level_values("structure_idx").nunique(),
)

# designs passing filters 242
# unique structures passing filters 59


Looks like we have a good number of designs meeting the ideal metric thresholds!

Next, we rank the designs based on iPAE to prioritize designs with high confidence of
interaction. We'll also select just the top design per unique structure, to select for
a diverse set of binders.

In [145]:
df_selected = (
    # rank by ipae
    df_filtered.reset_index().sort_values(by="ipae")
    # select best sequence per structure
    .groupby("structure_idx", sort=False).first()
    # set dataframe index
    .reset_index().set_index(["structure_idx", "sequence_idx"])
)
df_selected.head()

,,design_idx,rmsd,ipae,binder_rmsd,binder_plddt,score,sequence
structure_idx,sequence_idx,,,,,,,
32,7,327,1.156324,3.933931,0.517268,95.814362,1.0419,DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...
30,4,304,1.702794,4.004353,0.733918,93.942299,0.9605,DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...
99,8,998,0.980902,4.008550,0.745350,95.061966,1.0696,DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...
3,1,31,1.113197,4.215344,0.946744,93.851387,1.1261,DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...
73,9,739,0.896851,4.320190,0.608608,96.498856,1.1120,DYSFSCYSQLEVNGSQHSLTCAFEDPDVNTTNLEFEICGALVEVKC...


We now have a ranked list of promising binder designs!

Before we send them off for experimental validation, we should visually inspect their
stuctures for any anomalies. For example, the top ranked structure looks reasonable on
visual inspection:

In [160]:
design_idx, structure_idx = 327, 32
predicted_structure = predicted_structures[design_idx]
# TODO: run new rfdiffusion job so that transform here isn't needed
generated_structure = generated_structures[structure_idx].copy().transform(R=R)
predicted_structure = predicted_structure.copy().superimpose_onto(generated_structure)
visualize_cif(
    MolecularComplex(
        {
            "A_predicted": predicted_structure.get_protein("A"),
            "B_predicted": predicted_structure.get_protein("B"),
            "A_generated": generated_structure.get_protein("A"),
            "B_generated": generated_structure.get_protein("B"),
        }
    ).to_string(),
    colors=[
        ColorSpec(chain_id="A_predicted", color="#b5e2f5"),
        ColorSpec(chain_id="B_predicted", color="#f4c30b"),
        ColorSpec(
            chain_id="A_predicted",
            color="#6bb50a",
            positions=binding_sites,
            rep_type="ball_and_stick",
        ),
        ColorSpec(chain_id="A_generated", color="#F2F0EF"),
        ColorSpec(chain_id="B_generated", color="#F2F0EF"),
    ],
)

<IPython.core.display.Javascript object>

However, the fourth design may be undesirable in some cases because it consists of
exactly two alpha helices - previous works have found these structures to be less likely
to express in solution.

In [161]:
design_idx, structure_idx = 31, 3
predicted_structure = predicted_structures[design_idx]
# TODO: run new rfdiffusion job so that transform here isn't needed
generated_structure = generated_structures[structure_idx].copy().transform(R=R)
predicted_structure = predicted_structure.copy().superimpose_onto(generated_structure)
visualize_cif(
    MolecularComplex(
        {
            "A_predicted": predicted_structure.get_protein("A"),
            "B_predicted": predicted_structure.get_protein("B"),
            "A_generated": generated_structure.get_protein("A"),
            "B_generated": generated_structure.get_protein("B"),
        }
    ).to_string(),
    colors=[
        ColorSpec(chain_id="A_predicted", color="#b5e2f5"),
        ColorSpec(chain_id="B_predicted", color="#f4c30b"),
        ColorSpec(
            chain_id="A_predicted",
            color="#6bb50a",
            positions=binding_sites,
            rep_type="ball_and_stick",
        ),
        ColorSpec(chain_id="A_generated", color="#F2F0EF"),
        ColorSpec(chain_id="B_generated", color="#F2F0EF"),
    ],
)

<IPython.core.display.Javascript object>

After visually confirming the designs and further filtering based on any additional
metrics you may have in mind (e.g. metrics relevant to your specific assay), the designs
can then be sent off for experimental testing!

# Conclusion

In this tutorial, we've demonstrated how to design novel binders for a target of
interest. We validated the designs using in-silico metrics and visualized them to ensure
their viability. The top-ranked designs from this workflow can be:

1. Expressed and purified for experimental validation
1. Tested for binding affinity
1. Further optimized through additional rounds of design, for example, with
   [OpenProtein.AI's property regression models](https://docs.openprotein.ai/python-api/property-regression-models/index.html).